# md_form Tutorial: Form Parameters & UI Components

This notebook shows how to define form parameters with `md_form` and what each field type renders as in the UI. We'll recreate a simplified version of a transformation pipeline (inspired by intensity processing) to demonstrate the key concepts.

## 1. Setup

Install `md_form` if needed: `pip install -e .` (from the repo root). Then import the field helpers and utilities.

In [1]:
from pydantic import BaseModel
from md_form.field_utils import (
    select_field,
    number_field,
    boolean_field,
    string_field,
    numberrange_field,
    is_required,
    When,
)
from md_form import translate_payload

## 2. Define a Params Model

Define a Pydantic model for your form parameters. Each field uses an `md_form` helper that specifies:
- **fieldType**: which UI component to render (String, Number, Boolean, etc.)
- **name** / **description**: labels and help text
- **parameters**: options, constraints, or component-specific settings

In [2]:
class TransformIntensitiesParams(BaseModel):
    """Parameters for intensity transformation (simplified from an R pipeline)."""

    # SELECT FIELD → Dropdown in UI
    # Renders a dropdown with predefined options (like normalisation method picker)
    normalisation_method: str = select_field(
        default="none",
        options=["none", "scale", "quantile"],
        name="Normalisation Method",
        description="Method passed to limma normalizeBetweenArrays().",
    )

    # NUMBER FIELD → Number input
    # Renders a numeric input; ge/le set min/max constraints
    p_value_threshold: float = number_field(
        default=0.05,
        ge=0.0,
        le=1.0,
        name="P-value Threshold",
        description="Threshold for statistical significance.",
    )

    # BOOLEAN FIELD → Checkbox
    # Renders a checkbox; label can customise the text shown next to it
    apply_log_transform: bool = boolean_field(
        default=False,
        label="Apply log2 transform",
        name="Log Transform",
        description="Whether to log2-transform intensities.",
    )

    # STRING FIELD → Text input
    # Renders a single-line text field
    output_prefix: str = string_field(
        default="transformed",
        name="Output Prefix",
        description="Prefix for output dataset name.",
    )

    # NUMBER RANGE FIELD → Range slider
    # Renders a slider with min/max; interval controls step size
    intensity_range: float = numberrange_field(
        default=0.5,
        ge=0.0,
        le=1.0,
        interval=0.1,
        name="Intensity Filter Range",
        description="Keep intensities within this range (0–1).",
    )

## 3. Inspect the Schema

Each field helper adds metadata to the Pydantic model. The `json_schema_extra` on each field tells the UI which component to render and how to configure it.

In [3]:
import json

# Build the schema from the model
schema = TransformIntensitiesParams.model_json_schema()

# Show metadata for each param field
for key, val in schema.get("properties", {}).items():
    extra = val.get("json_schema_extra", {})
    if extra:
        print(f"\n--- {key} ---")
        print(json.dumps(extra, indent=2))

## 4. Field Type → UI Component Mapping

| `fieldType` | UI Component | Example |
|-------------|--------------|---------|
| `String` | Text input (single line) | `string_field`, `select_field` (with options) |
| `Number` | Number input / spinner | `number_field` |
| `NumberRange` | Range slider | `numberrange_field` |
| `Boolean` | Checkbox | `boolean_field` |
| `Datasets` | Dataset picker (INTENSITY type) | `intensity_input_dataset_field` |

`select_field` uses `fieldType: "String"` plus `parameters.options` — the UI renders it as a **dropdown** when options are present.

## 5. translate_payload

`translate_payload` converts a JSON-schema-like payload into a flattened form schema for UI rendering. It resolves refs, moves options/min/max into `parameters`, and keeps only UI-relevant keys.

In [4]:
# Full schema for dataset job (input_datasets + params) - required for translate_payload
params_schema = TransformIntensitiesParams.model_json_schema()
definitions = {
    "DatasetType": {"enum": ["INTENSITY", "DOSE_RESPONSE", "PAIRWISE", "ANOVA", "ENRICHMENT"], "title": "DatasetType", "type": "string"},
    "InputDatasetTable": {
        "properties": {"name": {"title": "Name", "type": "string"}, "bucket": {"default": None, "title": "Bucket", "type": "string"}, "key": {"default": None, "title": "Key", "type": "string"}, "data": {"default": None, "title": "Data"}},
        "required": ["name"], "title": "InputDatasetTable", "type": "object",
    },
    "IntensityInputDataset": {
        "properties": {
            "id": {"format": "uuid", "title": "Id", "type": "string"},
            "name": {"title": "Name", "type": "string"},
            "job_run_params": {"default": {}, "title": "Job Run Params", "type": "object"},
            "type": {"$ref": "#/definitions/DatasetType", "default": "INTENSITY"},
            "tables": {"items": {"$ref": "#/definitions/InputDatasetTable"}, "title": "Tables", "type": "array"},
        },
        "required": ["id", "name", "tables"], "title": "IntensityInputDataset", "type": "object",
    },
    "TransformIntensitiesParams": params_schema,
}
required_params_schema = {
    "properties": {
        "input_datasets": {
            "items": {"$ref": "#/definitions/IntensityInputDataset"},
            "maxItems": 1, "minItems": 1, "position": 0, "title": "input_datasets", "type": "array",
            "fieldType": "Datasets", "name": "Select Intensity dataset", "group": "Details",
            "parameters": {"multiple": False, "type": "INTENSITY", "width": "large"},
            "rules": [{"name": "is_required"}],
        },
        "params": {"$ref": "#/definitions/TransformIntensitiesParams", "position": 1, "title": "params"},
        "output_dataset_type": {"$ref": "#/definitions/DatasetType", "position": 2, "title": "output_dataset_type"},
    },
    "required": ["input_datasets", "params", "output_dataset_type"],
    "definitions": definitions,
}

# translate_payload flattens params → UI properties
properties = translate_payload(required_params_schema)

# Post-process: fix input_datasets name (overwritten by items flatten), add when/group
if "input_datasets" in properties and isinstance(properties["input_datasets"], dict):
    properties["input_datasets"]["name"] = "Select Intensity dataset"
when_dataset_present = {"is_present": True, "property": "input_datasets"}
param_keys = ["normalisation_method", "p_value_threshold", "apply_log_transform", "output_prefix", "intensity_range"]
for key in properties:
    if key != "input_datasets" and isinstance(properties.get(key), dict):
        properties[key]["when"] = when_dataset_present
        if key in param_keys:
            properties[key]["group"] = "Transform Parameters"

# Save for webapp rendering
import os
output_path = "transform_intensities_form.json"
with open(output_path, "w") as f:
    json.dump({"properties": properties}, f, indent=2)

print(f"Fields: {list(properties.keys())}")
print(f"Saved to {output_path}")

Fields: ['input_datasets', 'normalisation_method', 'p_value_threshold', 'apply_log_transform', 'output_prefix', 'intensity_range']
Saved to transform_intensities_form.json


## 6. Conditional Visibility (`when`) and Validation (`rules`)

Fields can be shown conditionally with `when`, and validated with `rules`:

In [5]:
# Example: a field that only appears when apply_log_transform is True
# and a required select with validation
log_base = number_field(
    default=2.0,
    ge=2.0,
    le=10.0,
    name="Log Base",
    description="Base for log transform (2 = log2, e = ln).",
    when=When.equals("apply_log_transform", True),
)

# Rules example: condition_column_field with validation
# (simplified - in a real form you'd use condition_column_field)
method = select_field(
    default="scale",
    options=["none", "scale", "quantile"],
    name="Method",
    description="Normalisation method.",
    rules=is_required(),
)

# Inspect the when condition on log_base
log_base_info = log_base.json_schema_extra
print("Field with conditional visibility:")
print("  when:", log_base_info.get("when"))
print("\nField with validation rule:")
print("  rules:", method.json_schema_extra.get("rules"))

Field with conditional visibility:
  when: {'property': 'apply_log_transform', 'equals': True}

Field with validation rule:
  rules: {'name': 'is_required'}


## 7. Create and Validate an Instance

You can instantiate the model with values; Pydantic validates against the constraints (ge, le, options, etc.).

In [6]:
# Valid instance
params = TransformIntensitiesParams(
    normalisation_method="quantile",
    p_value_threshold=0.01,
    apply_log_transform=True,
    output_prefix="my_transformed",
    intensity_range=0.3,
)
print("Valid params:", params.model_dump())

# Invalid: would raise ValidationError (try uncommenting)
# bad = TransformIntensitiesParams(normalisation_method="invalid_option")

Valid params: {'normalisation_method': 'quantile', 'p_value_threshold': 0.01, 'apply_log_transform': True, 'output_prefix': 'my_transformed', 'intensity_range': 0.3}


## 8. Example 2: Entity-based conditional filtration (PTM only for peptide)

This example mirrors the pattern from `intensity_imputation_types.py` in md-converter:
- **entity_type**: peptide | protein | gene
- **filtration_methods**: "none" | "ptm_localization_probability"
- **PTMLocalisationFilterParams** (the ptm option) should only appear when `entity_type` is **peptide**
- The **threshold** field appears when filtration method is "ptm_localization_probability"

In [7]:
# Example 2: Entity-based conditional filtration (like intensity_imputation_types.py)
# PTMLocalisationFilterParams shows only when entity_type is "peptide"

from typing import Union, Literal

# --- Filtration variants (discriminated union) ---

class FiltrationNoneParams(BaseModel):
    """Skip filtration - always available."""
    method: Literal["none"] = select_field(
        name="Filtration Method",
        default="none",
        options=["none"],
        rules=[is_required()],
    )


class PTMLocalisationFilterParams(BaseModel):
    """PTM Localisation filter - only for peptide entity type."""
    method: Literal["ptm_localization_probability"] = select_field(
        name="Filtration Method",
        default="ptm_localization_probability",
        options=["ptm_localization_probability"],
        rules=[is_required()],
        when=When.equals("entity_type", "peptide"),  # Only show this option when entity_type is peptide
    )
    threshold: float = numberrange_field(
        default=0.5,
        ge=0.0,
        le=1.0,
        interval=0.01,
        name="PTM Localisation Filter Threshold",
        description="Filter PTM sites by PTMProbMax ≥ threshold.",
        when=When.equals("filtration_methods", "ptm_localization_probability"),
    )


class EntityTypeFiltrationParams(BaseModel):
    """Params with entity_type and conditional filtration (mirrors NormalisationAndImputationParams pattern)."""
    entity_type: Literal["peptide", "protein", "gene"] = select_field(
        name="Entity Type",
        default="peptide",
        options=["peptide", "protein", "gene"],
        description="Entity type of the intensity dataset (peptide enables PTM options).",
        rules=[is_required()],
    )
    filtration_methods: Union[FiltrationNoneParams, PTMLocalisationFilterParams] = select_field(
        name="Filtration Methods",
        discriminator="method",
        options=["none", "ptm_localization_probability"],
        default="none",
        rules=[is_required()],
    )


# Inspect the schema for the PTM variant
ptm_schema = PTMLocalisationFilterParams.model_json_schema()
print("PTMLocalisationFilterParams - method field has when:")
print(json.dumps(ptm_schema.get("properties", {}).get("method", {}).get("json_schema_extra", {}), indent=2))

PTMLocalisationFilterParams - method field has when:
{}


In [8]:
# Build form schema for Example 2 (EntityTypeFiltrationParams)
# Structure: params ref only - no input_datasets for this tutorial

params_schema_e2 = EntityTypeFiltrationParams.model_json_schema()
definitions_e2 = {"EntityTypeFiltrationParams": params_schema_e2}
required_schema_e2 = {
    "properties": {
        "params": {"$ref": "#/definitions/EntityTypeFiltrationParams", "position": 0, "title": "params"},
    },
    "required": ["params"],
    "definitions": definitions_e2,
}

properties_e2 = translate_payload(required_schema_e2)
print("Flattened fields:", list(properties_e2.keys()))

# Post-process: add conditional options for PTM
# The "ptm_localization_probability" option should only show when entity_type is peptide.
# Some form UIs support when on individual options:
if "filtration_methods" in properties_e2 and "parameters" in properties_e2["filtration_methods"]:
    opts = properties_e2["filtration_methods"]["parameters"].get("options", [])
    # Add when to the ptm option so it only appears when entity_type is peptide
    new_opts = []
    for o in opts:
        if isinstance(o, dict):
            opt = dict(o)
            if opt.get("value") == "ptm_localization_probability":
                opt["when"] = {"property": "entity_type", "equals": "peptide"}
            new_opts.append(opt)
        else:
            new_opts.append(o)
    properties_e2["filtration_methods"]["parameters"]["options"] = new_opts

# Save for inspection (same dir as notebook when run from tutorial/)
with open("entity_filtration_form.json", "w") as f:
    json.dump({"properties": properties_e2}, f, indent=2)
print("Saved entity_filtration_form.json")
print("\nfiltration_methods.options with conditional when on PTM:")
print(json.dumps(properties_e2.get("filtration_methods", {}).get("parameters", {}).get("options", []), indent=2))

Flattened fields: ['entity_type', 'filtration_methods', '$defs']
Saved entity_filtration_form.json

filtration_methods.options with conditional when on PTM:
[
  {
    "name": "none",
    "value": "none"
  },
  {
    "name": "ptm_localization_probability",
    "value": "ptm_localization_probability",
    "when": {
      "property": "entity_type",
      "equals": "peptide"
    }
  }
]


### Testing conditional visibility in the notebook

You can test the conditional logic by:

1. **Valid combinations**: `entity_type=peptide` + `filtration_methods=ptm_localization_probability` + `threshold=0.5`
2. **Valid for protein/gene**: `entity_type=protein` + `filtration_methods=none` (PTM option hidden in UI)
3. **Simulate form state**: Check that `when` conditions evaluate as expected

In [9]:
# Test 1: Valid params when entity_type is peptide and PTM filter is used
params_peptide_ptm = EntityTypeFiltrationParams(
    entity_type="peptide",
    filtration_methods={"method": "ptm_localization_probability", "threshold": 0.75},
)
print("✓ entity_type=peptide, filtration=ptm_localization_probability:")
print(" ", params_peptide_ptm.model_dump())

# Test 2: Valid params when entity_type is protein - only "none" filtration
params_protein_none = EntityTypeFiltrationParams(
    entity_type="protein",
    filtration_methods={"method": "none"},
)
print("\n✓ entity_type=protein, filtration=none:")
print(" ", params_protein_none.model_dump())

# Test 3: Simulate when-condition evaluation (for UI testing)
def when_matches(form_values: dict, when_spec: dict) -> bool:
    """Simple evaluator: property equals value."""
    if "property" in when_spec and "equals" in when_spec:
        return form_values.get(when_spec["property"]) == when_spec["equals"]
    return True

# When entity_type=peptide → PTM option should show
form_peptide = {"entity_type": "peptide", "filtration_methods": "ptm_localization_probability"}
ptm_option_when = {"property": "entity_type", "equals": "peptide"}
print("\n✓ Form state entity_type=peptide → PTM option visible:", when_matches(form_peptide, ptm_option_when))

# When entity_type=protein → PTM option should be hidden
form_protein = {"entity_type": "protein"}
print("✓ Form state entity_type=protein → PTM option visible:", when_matches(form_protein, ptm_option_when))

✓ entity_type=peptide, filtration=ptm_localization_probability:
  {'entity_type': 'peptide', 'filtration_methods': {'method': 'ptm_localization_probability', 'threshold': 0.75}}

✓ entity_type=protein, filtration=none:
  {'entity_type': 'protein', 'filtration_methods': {'method': 'none'}}

✓ Form state entity_type=peptide → PTM option visible: True
✓ Form state entity_type=protein → PTM option visible: False
